# Supply-Chain Optimization with NVIDIA cuOpt (GPU) + Pyomo (CPU)

This notebook layers three classic operations-research problems on top of a generated complex-tier supply-chain network. A **`company` widget** selects which graph to use:

- **`mps`** — the illustrative Monolithic Power Systems fabless-PMIC network (`dataset_realistic_mps.json`), the pipeline's first example.
- **`apple_real`** — a network grounded in Apple's **published** supplier list (`dataset_realistic_apple_real.json`; see `scripts/apple_supplier_list.py`), with real supplier names + regions.

MPS was just the first example — the FJSP/CVRPTW/MEIO derivation is company-parameterized via a `DerivationConfig` (`scripts/mps_derivation.py`), so the same engine runs on either graph. A `scale` widget runs it at complex (~2,300 nodes) or planet (~76,000 nodes) scale.

| Scenario | Problem | Solver | Compute |
|---|---|---|---|
| **II — FJSP** | Flexible Job-Shop Scheduling of back-end assembly/test across the company's assembly/OSAT machines, minimizing makespan | NVIDIA cuOpt MILP (Feasibility Pump / PDLP) | **GPU** |
| **III — CVRPTW** | Capacitated Vehicle Routing with Time Windows for finished-goods distribution from the company's dispatch depots | NVIDIA cuOpt routing (GES) | **GPU** |
| **I — MEIO** | Multi-Echelon Inventory Optimization (Guaranteed Service Model) across the whole tier1/tier2/tier3 network | Pyomo + HiGHS (piecewise-linear) | **CPU** |

The FJSP and CVRPTW problems are **derived from the generated network** (via `scripts/mps_derivation.py`), not invented from scratch — FJSP machines come from the company's back-end/assembly tier-2 anchors, jobs from tier-1 product-line demand, customers from tier-1 lines, depots from the company's dispatch hubs. The MEIO network is the full generated graph with its calibrated holding-cost / lead-time fields.

> ### ⚠️ Support caveat — read before running on GPU
>
> Running **NVIDIA cuOpt on Databricks serverless GPU compute is technically feasible but is NOT officially supported or documented by Databricks or NVIDIA** as of this writing. The cuOpt wheels are installed from NVIDIA's own package index (`https://pypi.nvidia.com`), not from the Databricks runtime. Treat the GPU sections here as an integration experiment: they are gated behind a `run_gpu` widget (default `"no"`) so the CPU-verifiable parts (the derivation layer and the MEIO solve) run end-to-end on any cluster, GPU or not.

> **Disclaimer:** every numeric figure attached to a node — margin, inventory, demand, capacity, cost, lead time — is synthetic/illustrative (see `scripts/company_profiles.py`). For `apple_real` the supplier **names and regions** are real/published, but the numbers and the tier/role assignment are still a model.

## Cluster Configuration

**CPU-only sections (derivation + MEIO)** run on the same single-node config as `05`–`08`:
- **Databricks Runtime Version:** 17.3 LTS ML
- **Single Node** — Azure `Standard_DS4_v2` / AWS `m5d.2xlarge`

**GPU sections (FJSP + CVRPTW, `run_gpu="yes"`)** require an NVIDIA GPU with CUDA 12+:
- **Databricks serverless GPU compute** (AI Runtime) — A10 (24 GB), H100 (80 GB), or 8×H100.
- **Status:** serverless GPU is **Public Preview**, single-node only, 7-day max runtime, and available in only a subset of AWS regions. See <https://docs.databricks.com/aws/en/machine-learning/ai-runtime/>.
- cuOpt itself requires NVIDIA Volta+ (compute capability ≥ 7.0), CUDA 12.0+, Python 3.11–3.14, OpenSSL 3, and Linux/WSL2 — all satisfied by the serverless GPU image, but again this specific combination is not a documented/supported Databricks configuration.

In [ ]:
dbutils.widgets.dropdown("run_gpu", "no", ["no", "yes"], "Run GPU (cuOpt) sections")
run_gpu = dbutils.widgets.get("run_gpu") == "yes"

dbutils.widgets.dropdown("company", "mps", ["mps", "apple_real"], "Company (supply-chain graph)")
company = dbutils.widgets.get("company")

dbutils.widgets.dropdown("scale", "complex", ["complex", "planet"], "Network scale")
scale = dbutils.widgets.get("scale")

print(f"run_gpu={run_gpu}, company={company}, scale={scale}")
print(
    "GPU FJSP/CVRPTW cells will run." if run_gpu
    else "GPU cells will be SKIPPED; the derivation + MEIO (CPU) sections still run end-to-end."
)
if scale == "planet":
    print(
        "scale='planet' (~76k nodes): FJSP machines / CVRPTW depots stay scale-invariant "
        "(keyed off the company's real named anchors), and MEIO subsamples the network "
        "to stay tractable on CPU — see the MEIO section."
    )

In [ ]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

### GPU dependency install (only when `run_gpu="yes"`)

The cell below installs the cuOpt/cuDF wheels from NVIDIA's package index (`requirements-gpu.txt`). It is a no-op when `run_gpu="no"`, so this notebook installs nothing GPU-specific on a CPU cluster. This is the step that is **not officially supported** on Databricks (see the caveat at the top).

In [ ]:
run_gpu = dbutils.widgets.get("run_gpu") == "yes"  # re-read after restartPython
if run_gpu:
    %pip install -r ./requirements-gpu.txt --extra-index-url=https://pypi.nvidia.com --quiet
    dbutils.library.restartPython()
else:
    print("run_gpu='no' — skipping cuOpt/cuDF install.")

In [ ]:
import json
import random

import scripts.realistic_topologies as rt
import scripts.scenario_calibration as sc
import scripts.mps_derivation as md
import scripts.meio_pyomo as meio
import scripts.fjsp_cuopt as fjsp
import scripts.cvrptw_cuopt as cvrptw

run_gpu = dbutils.widgets.get("run_gpu") == "yes"
company = dbutils.widgets.get("company")
scale = dbutils.widgets.get("scale")

# The derivation pipeline is company-parameterized: pick the DerivationConfig
# that tells select_backend_machines / select_cvrptw_depots / derive_meio_network
# which real anchors + material-type roles to key off. MPS was the first
# example; apple_real is grounded in Apple's published supplier list.
DERIVATION_CONFIGS = {"mps": md.MPS_CONFIG, "apple_real": md.APPLE_REAL_CONFIG}
config = DERIVATION_CONFIGS[company]
print(f"Using {company} derivation config ({len(config.anchor_names)} named anchors, "
      f"machine roles={sorted(config.machine_material_tags)})")

## Load the network

We load the dataset for the selected `company` + `scale` (written by `05_realistic_operational_data`). If it isn't on the volume yet (e.g. you're running `09` standalone), we regenerate it in-memory. We then merge in the calibrated cost/holding-cost/lead-time/emissions fields (the same `calibrate_cost_fields` the objective-function notebooks use, keyed to the company's industry profile) — the FJSP processing times and MEIO holding costs read from these.

In [ ]:
catalog = "supply_chain_stress_test"  # Change here (match 05)
schema = "data"                       # Change here
volume = "operational"                # Change here
volume_dir = f"/Volumes/{catalog}/{schema}/{volume}"

company = dbutils.widgets.get("company")  # re-read after restartPython
scale = dbutils.widgets.get("scale")
suffix = "" if scale == "complex" else "_planet"
dataset_filename = f"dataset_realistic_{company}{suffix}.json"
dataset_path = f"{volume_dir}/{dataset_filename}"

import os
if os.path.exists(dataset_path):
    with open(dataset_path) as f:
        net_ds = json.load(f)
    print(f"Loaded {company} ({scale}) dataset from {dataset_path}")
else:
    if scale == "planet":
        net_ds = rt.generate_complex_network_at_scale(company, scale="planet")
    else:
        net_ds = rt.generate_complex_network(company)
    print(f"{company} ({scale}) dataset not found on volume — regenerated in-memory (run 05 to persist it).")

# Merge calibrated cost/holding/lead-time/emissions fields (industry profile
# keyed to the company). FJSP processing times + MEIO holding costs read these.
_fields = sc.calibrate_cost_fields(
    random.Random(7),
    net_ds["tier1"], net_ds["tier2"], net_ds["tier3"],
    net_ds["material_types"], net_ds["supplier_material_type"],
    net_ds["edges"], net_ds.get("criticality"), company,
    region=net_ds.get("region"),
)
net_ds = {**net_ds, **_fields}
print(f"tier1={len(net_ds['tier1'])}, tier2={len(net_ds['tier2'])}, tier3={len(net_ds['tier3'])}")

In [ ]:
rt.visualize_anchor_backbone(net_ds)

## Scenario II — FJSP (back-end scheduling on GPU)

We derive **machines** from the company's back-end/assembly tier-2 anchors (via the selected `DerivationConfig` — for MPS: Chengdu wafer-sort + OSAT lines; for apple_real: final-assembly EMS + OSAT), **jobs** from tier-1 product-line demand (each job is a lot flowing through the config's operation sequence), and **processing times** from each facility's calibrated `production_delay`. The config's `test_operation` is restricted to test-capable machines. Because machines are keyed off the real named anchors, the machine count is **scale-invariant** (same at complex and planet scale).

The derivation below is pure Python and runs on any cluster. The `build_fjsp_problem`/`solve_fjsp` calls need the cuOpt GPU wheel and only run when `run_gpu="yes"`.

In [ ]:
machines = md.select_backend_machines(net_ds, config)
jobs = md.derive_fjsp_jobs(net_ds, machines, config=config)
processing_times = md.derive_fjsp_processing_times(jobs, machines, net_ds, config=config)

spec = fjsp._fjsp_milp_spec(jobs, machines, processing_times)
print(f"{len(machines)} machines, {len(jobs)} jobs, {len(processing_times)} (job,op,machine) processing-time entries")
print(f"MILP: {spec.n_vars} variables ({spec.n_binary} binary), {spec.n_constraints} constraints, big_M={spec.big_m:.1f}")
from collections import Counter
print("machine groups:", dict(Counter(m['group'] for m in machines)))
print("operations:", config.operations)

In [ ]:
if run_gpu:
    problem, spec = fjsp.build_fjsp_problem(jobs, machines, processing_times)
    raw = fjsp.solve_fjsp(problem, time_limit=60.0)
    schedule = fjsp.decode_fjsp_result(raw, jobs, machines, processing_times)
    print(f"makespan = {schedule['makespan']}")
    # Show the first few machine timelines
    for mid, ops in list(schedule["by_machine"].items())[:5]:
        if ops:
            timeline = ", ".join(f"{a['job']}/{a['operation']}[{a['start']}-{a['end']}]" for a in ops)
            print(f"  {mid}: {timeline}")
else:
    print("run_gpu='no' — skipping the cuOpt FJSP GPU solve. The MILP spec above was still built on CPU.")

## Scenario III — CVRPTW (distribution routing on GPU)

We derive **depots** from the Chengdu/Penang facilities, one **customer** per tier-1 product line (tagged with an illustrative OEM-destination region and a demand volume), **time windows** that tighten for customers fed predominantly by `monopoly_bottleneck` suppliers, and a synthetic **distance/time matrix** from a region-pair lookup (no real geocoding). Again the derivation runs anywhere; the `build_cvrptw_datamodel`/`solve_cvrptw` calls need cuOpt + cuDF and only run when `run_gpu="yes"`.

In [ ]:
depots = md.select_cvrptw_depots(net_ds, config)
customers = md.derive_cvrptw_customers(net_ds)
time_windows = md.derive_cvrptw_time_windows(customers, net_ds)
distance_matrix = md.derive_cvrptw_distance_time_matrix(depots, customers)
vehicles = {"n_vehicles": 4, "capacity": 10_000}

cvrptw_spec = cvrptw._cvrptw_datamodel_spec(depots, customers, vehicles, distance_matrix, time_windows)
cvrptw_spec.validate()
print(f"{len(depots)} depots, {len(customers)} customers, {cvrptw_spec.n_vehicles} vehicles")
print("depots:", [d['name'] for d in depots])
for c in customers:
    lo, hi = time_windows[c['customer_id']]
    print(f"  {c['customer_id']} {c['product_line']} -> {c['region']}, demand={c['demand']}, window=[{lo},{hi}]")

In [ ]:
if run_gpu:
    data_model, cvrptw_spec = cvrptw.build_cvrptw_datamodel(
        depots, customers, vehicles, distance_matrix, time_windows
    )
    raw = cvrptw.solve_cvrptw(data_model, time_limit=10.0)
    routes = cvrptw.decode_cvrptw_result(raw, cvrptw_spec)
    print(f"status={routes['status']}, total_cost={routes['total_cost']}, vehicles used={routes['n_vehicles_used']}")
    for vid, stops in routes["routes"].items():
        print(f"  vehicle {vid}: {' -> '.join(map(str, stops))}")
else:
    print("run_gpu='no' — skipping the cuOpt CVRPTW GPU solve. The routing model spec above was still built/validated on CPU.")

## Scenario I — MEIO (multi-echelon inventory, CPU)

MEIO stays **CPU-only** by design (see the caveat at the top and `docs/mps-cuopt-pipeline.md`): the Guaranteed Service Model's true objective has a square-root nonlinearity that the repo's HiGHS (LP/MILP-only) stack can't solve directly, so `scripts/meio_pyomo.py` approximates `sqrt(net_lead_time)` with a piecewise-linear interpolant (SOS2-style, via Pyomo's incremental `Piecewise` representation) — keeping it a MILP HiGHS *can* solve, with no new dependency. This section always runs, regardless of `run_gpu`.

**At `scale="planet"`** the full ~76,000-node network would produce ~76k piecewise-linear MILP blocks, which HiGHS can't solve in a reasonable time. `derive_meio_network(mps, max_nodes=...)` subsamples the network first — always keeping every tier-1 node, MPS's real named anchors, and every `monopoly_bottleneck`/`oligopoly` node (the fragile, capital-intensive nodes where safety-stock placement actually matters), and randomly sampling the commodity long tail. At `scale="complex"` the full network is solved (`max_nodes=None`).

In [ ]:
# At planet scale the full ~76k-node GSM MILP is intractable in HiGHS (one
# INC piecewise block per node). Subsample to a tractable size while ALWAYS
# keeping tier-1 nodes, the company's named anchors, and every monopoly/
# oligopoly node (where safety-stock placement actually matters). At complex
# scale we solve the full network (max_nodes=None).
meio_max_nodes = None if scale == "complex" else 2500
meio_network = md.derive_meio_network(net_ds, max_nodes=meio_max_nodes, config=config)
model = meio.build_meio_model(meio_network, mode="piecewise_linear")
result = meio.solve_meio(model)

print(f"company={company}, scale={scale}, MEIO nodes={len(meio_network['nodes'])}")
print(f"termination: {result['termination_condition']}")
print(f"total safety-stock cost (piecewise-linear approx): {result['total_safety_stock_cost']:.2f}")

# Bound the approximation error against the exact true-sqrt objective.
true_cost = meio.true_safety_stock_cost(meio_network, result["per_node"])
rel_err = abs(result["total_safety_stock_cost"] - true_cost) / max(1e-9, true_cost)
print(f"total safety-stock cost (exact sqrt):            {true_cost:.2f}")
print(f"piecewise-linear approximation error:            {rel_err:.2%}")

In [ ]:
# Where does the model concentrate safety stock? Show the top nodes by cost.
top = sorted(result["per_node"].items(), key=lambda kv: kv[1]["safety_stock_cost"], reverse=True)[:10]
print("Top 10 nodes by safety-stock cost:")
for node, info in top:
    name = net_ds.get("company_name", {}).get(node) or node
    crit = net_ds.get("criticality", {}).get(node, "generic")
    print(f"  {name} ({crit}): cost={info['safety_stock_cost']:.2f}, net_lead_time={info['net_lead_time']:.2f}")

## Non-goals (v1 scope)

This notebook computes the FJSP / CVRPTW / MEIO results **directly**. It deliberately does **not** implement the full 5-stage agentic pipeline from `MPS Supply Chain Optimization Pipeline.md` §8 (Adexa digital-twin ingestion → NVIDIA NIM/LLM reasoning → LangChain sub-agents that formulate the matrices → cuOpt GPU solve → Adexa S&OE execution). That orchestration layer — which would naturally reuse this repo's existing `Supply_Chain_Supervisor` MAS/Genie/MCP infrastructure to turn a natural-language planner query into these three model runs — is a documented **future extension**, not part of v1.

Other v1 simplifications, all documented where they occur:
- Chengdu and Penang (MPS-owned captive facilities) are modeled as tier-2 "supplier" nodes because the LP schema only has tier1/tier2/tier3 slots.
- OSAT partners use generic labels (the source doc doesn't name them).
- The CVRPTW distance/time matrix is a coarse region-pair lookup, not real geography.
- MEIO uses a piecewise-linear approximation of the sqrt safety-stock term rather than a true nonlinear (MINLP) solve — the exact nonlinear path is a stubbed `mode="nonlinear"` that would need an Ipopt-class solver.
- `09` is **not** part of the `05`→`08` Databricks multi-task Job graph (it's GPU-optional and separate).

## Wrap Up

We derived three operations-research problems from the generated MPS-like network and solved them: **FJSP** (back-end assembly/test scheduling) and **CVRPTW** (distribution routing) on NVIDIA cuOpt's GPU solvers when `run_gpu="yes"`, and **MEIO** (multi-echelon safety-stock placement) on Pyomo/HiGHS CPU always. The GPU sections are an unofficial cuOpt-on-Databricks-serverless-GPU integration (see the caveat at the top); the derivation layer and MEIO are fully reproducible on any cluster. See `docs/mps-cuopt-pipeline.md` for the formulations, the derivation logic, and the support caveat in full.

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source]. All included or referenced third party libraries are subject to the licenses set forth below.

| library | description | license | source |
|---|---|---|---|
| pyomo | Algebraic modeling language for optimization | BSD-3 | https://pypi.org/project/pyomo/ |
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/ |
| cuopt-cu12 | NVIDIA GPU-accelerated decision-optimization engine | NVIDIA proprietary (license terms pending verification — confirm at https://pypi.org/project/cuopt-cu12/ before redistribution) | https://pypi.org/project/cuopt-cu12/ |
| cudf-cu12 | NVIDIA RAPIDS GPU DataFrame library | Apache 2.0 (confirm at source before redistribution) | https://pypi.org/project/cudf-cu12/ |